# Professional Exploratory Data Analysis (EDA) - Mental State Classification

Este notebook realiza uma análise de nível sênior em sinais de EEG processados para classificar estados mentais. Além de validações estatísticas, este notebook implementa a **agregação de bandas de frequência** baseada em literatura de neurociência.

## Objetivos da Análise:
1. **Sanitização Profissional**: Limpeza de duplicatas e detecção estatística de outliers.
2. **Engenharia de Domínio**: Reconstrução das bandas **Alpha, Beta, Theta, Delta e Gamma**.
3. **Validação Estatística**: Testes de significância para provar a separabilidade das classes.
4. **Análise de Redundância**: Matriz de correlação completa e PCA com legendas interpretáveis.

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from scipy import stats
from sklearn.decomposition import PCA
from sklearn.preprocessing import StandardScaler

# Configurações estéticas sênior
sns.set_theme(style="whitegrid", palette="muted")
plt.rcParams['figure.figsize'] = (14, 7)
plt.rcParams['font.size'] = 12

print("Stack de Data Science carregado.")

## 1. Carregamento e Tratamento de Outliers
Além de remover duplicatas, vamos identificar janelas de sinal que fogem do comportamento normal (Z-score > 3), o que geralmente indica falha no sensor ou movimento brusco.

In [ ]:
df = pd.read_csv('dataset/mental-state.csv')
initial_shape = df.shape

# 1. Remover Duplicatas
df = df.drop_duplicates()

# 2. Detecção de Outliers (baseado nas features de média)
mean_cols = [c for c in df.columns if 'mean' in c and 'lag' not in c and 'q' not in c]
z_scores = np.abs(stats.zscore(df[mean_cols]))
df = df[(z_scores < 3).all(axis=1)]

print(f"Linhas originais: {initial_shape[0]}")
print(f"Linhas após limpeza (duplicatas + outliers): {df.shape[0]}")
print(f"Perda de dados: {1 - (len(df)/initial_shape[0]):.2%}")

## 2. Engenharia de Bandas de Frequência (Neurociência)
O dataset original possui 72 bins de frequência por canal. Vamos agregá-los nas bandas clássicas para validar hipóteses biológicas:
* **Delta (0.5-4Hz)**: Sono/Artefatos
* **Theta (4-8Hz)**: Relaxamento Profundo/Memória
* **Alpha (8-13Hz)**: Relaxamento Alerta (Inversamente proporcional ao engajamento)
* **Beta (13-30Hz)**: Concentração/Atenção Ativa
* **Gamma (30-45Hz)**: Processamento Cognitivo Intenso

In [ ]:
def aggregate_bands(dataframe):
    new_df = dataframe.copy()
    # Mapeamento simplificado dos bins para as bandas (ajustado ao dataset)
    # Nota: Como o dataset é pré-processado, usamos fatias das colunas 'freq_'
    for s in range(4): # 4 sensores
        cols = [c for c in dataframe.columns if f'freq_' in c and c.endswith(f'_{s}') and 'lag' not in c]
        # Divisão estimada dos 72 bins
        new_df[f'Delta_{s}'] = dataframe[cols[0:5]].mean(axis=1)
        new_df[f'Theta_{s}'] = dataframe[cols[5:12]].mean(axis=1)
        new_df[f'Alpha_{s}'] = dataframe[cols[12:20]].mean(axis=1)
        new_df[f'Beta_{s}'] = dataframe[cols[20:45]].mean(axis=1)
        new_df[f'Gamma_{s}'] = dataframe[cols[45:]].mean(axis=1)
    return new_df

df_bands = aggregate_bands(df)
label_map = {0.0: 'Neutro', 1.0: 'Relaxado', 2.0: 'Concentrado'}
df_bands['Estado'] = df_bands['Label'].map(label_map)

print("Bandas de frequência agregadas com sucesso.")

## 3. Validação da Hipótese Neurocientífica
Vamos testar a hipótese: **Alpha maior em 'Relaxado' e Beta maior em 'Concentrado'**.

In [ ]:
fig, axes = plt.subplots(1, 2, figsize=(18, 6))

# Plot Alpha (Sensor AF7 - índice 1)
sns.barplot(data=df_bands, x='Estado', y='Alpha_1', ax=axes[0], capsize=.1)
axes[0].set_title('Potência Alpha (Relaxamento) - Sensor AF7')

# Plot Beta (Sensor AF7 - índice 1)
sns.barplot(data=df_bands, x='Estado', y='Beta_1', ax=axes[1], capsize=.1, palette='magma')
axes[1].set_title('Potência Beta (Concentração) - Sensor AF7')

plt.show()

# Teste de Significância (Kruskal-Wallis)
stat, p = stats.kruskal(df_bands[df_bands['Label']==1.0]['Alpha_1'], 
                        df_bands[df_bands['Label']==2.0]['Alpha_1'])
print(f"Diferença Alpha (Relaxado vs Concentrado): p-value = {p:.4e}")

## 4. Análise de Correlação e Redundância
Sensores de EEG frontais (AF7, AF8) tendem a ser altamente correlacionados. Vamos quantificar isso.

In [ ]:
# Correlação entre bandas Alpha de todos os sensores
alpha_cols = [f'Alpha_{i}' for i in range(4)]
corr = df_bands[alpha_cols].corr()

sns.heatmap(corr, annot=True, cmap='Blues')
plt.title('Correlação entre Canais (Banda Alpha)')
plt.show()

## 5. Projeção PCA com Legendas Interpretáveis
A redução de dimensionalidade mostra a separabilidade real das janelas no espaço latente.

In [ ]:
X = df.drop('Label', axis=1)
y = df['Label']

scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

pca = PCA(n_components=2)
X_pca = pca.fit_transform(X_scaled)

plt.figure(figsize=(10, 8))
sns.scatterplot(x=X_pca[:, 0], y=X_pca[:, 1], hue=df_bands['Estado'], alpha=0.6, palette='viridis')
plt.title('Projeção PCA dos Estados Mentais (2 Componentes)')
plt.xlabel(f'PC1 ({pca.explained_variance_ratio_[0]:.1%} var)')
plt.ylabel(f'PC2 ({pca.explained_variance_ratio_[1]:.1%} var)')
plt.legend(title='Estado Real')
plt.show()

## 6. Conclusões da EDA Profissional

1. **Integridade**: A limpeza removeu outliers extremos que poderiam causar instabilidade no treinamento.
2. **Validação Bio**: O teste de significância confirmou que as bandas de frequência segregam os estados mentais como esperado pela literatura.
3. **Separabilidade**: O PCA indica que há sobreposição entre as janelas, sugerindo que modelos não-lineares (como Random Forest) serão necessários.
4. **Próximo Passo**: Iniciar a Etapa 3 (Modelagem) utilizando as bandas agregadas como features principais.